# Lung - TabPFN-2.5 || TabICL

In [ ]:
# ============================================================
# Lung multiclass
# Fixed best GO-LR + NSC hyperparameters
# Compare:
#   (A) NSC -> TabPFN-2.5
#   (B) NSC -> TabICL
# Eval:
#   5x5 CV with SAME folds and SAME compressed features
# Metrics:
#   Accuracy, ROC-AUC (macro OVR), Macro-F1, Runtime
# GPU:
#   cuda:7
# ============================================================

import os
# ---- set CUDA before torch import ----
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import gc
import time
import random
import inspect
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.utils.validation import check_X_y, check_array

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config
PIDFSegPCA = pidf_segpca

from tabicl import TabICLClassifier
from tabicl.sklearn.preprocessing import (
    EnsembleGenerator,
    UniqueFeatureFilter,
    PreprocessingPipeline,
    CustomStandardScaler,
    OutlierRemover,
    TransformToNumerical,
)

# ============================================================
# Config
# ============================================================
SEED = 42
DATA_FILE = "lung_combined_encoded.csv"
TARGET_COL = "Label"

BEST_PARAMS = {
    "go_metric": "manhattan",
    "go_num_clusters": 11,
    "go_refine_passes": 1,
    "go_direction_select": True,
    "nsc_segmentation": "uniform",
    "nsc_m_rule": "default",
    "nsc_tau": 0.99,
    "nsc_gamma": 2.3798196044385556,
    "nsc_beta": 0.7926636166665593,
    "nsc_Mmin": 64,
    "nsc_Mmax": 384,
    "nsc_lmin": 12,
    "assume_standardized": False,
    "tabpfn_seed": 42,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# Utils
# ============================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()

def ensure_multiclass_contiguous(y: np.ndarray):
    y = np.asarray(y).reshape(-1)
    uniq = np.unique(y)
    uniq_sorted = np.sort(uniq)
    class_map = {orig: i for i, orig in enumerate(uniq_sorted.tolist())}
    y_enc = np.vectorize(class_map.get, otypes=[np.int64])(y).astype(np.int64)
    return y_enc, class_map, int(len(class_map))

def compute_deltas_adjacent_corr(X_tr: np.ndarray, Pi_star: list, eps: float = 1e-12) -> torch.Tensor:
    X = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)
    Xp = X[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std
    corr = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    return (1.0 - corr.abs()).cpu()

def compute_multiclass_metrics(y_true, proba, pred, n_classes):
    proba = np.asarray(proba)
    acc = float(accuracy_score(y_true, pred))
    auc = float(roc_auc_score(y_true, proba, multi_class="ovr", average="macro"))
    f1  = float(f1_score(y_true, pred, average="macro"))
    return acc, auc, f1

# ============================================================
# Patch TabICL internals
# ============================================================
def _set_feature_metadata(est, X):
    X_arr = X
    if hasattr(X, "values"):
        X_arr = X.values
    est.n_features_in_ = X_arr.shape[1]
    if hasattr(X, "columns"):
        est.feature_names_in_ = np.array(X.columns, dtype=object)

def _clf_validate_data(self,
                       X,
                       y=None,
                       dtype=None,
                       cast_to_ndarray=False,
                       skip_check_array=False,
                       **kwargs):
    if skip_check_array:
        _set_feature_metadata(self, X)
        return X, y

    if isinstance(X, pd.DataFrame):
        cols = X.columns
        X_out, y_out = check_X_y(X.values, y, accept_sparse=False, dtype=dtype)
        X_df = pd.DataFrame(X_out, columns=cols)
        _set_feature_metadata(self, X_df)
        return X_df, y_out

    X_out, y_out = check_X_y(X, y, accept_sparse=False, dtype=dtype)
    _set_feature_metadata(self, X_out)
    return X_out, y_out

TabICLClassifier._validate_data = _clf_validate_data

def _eg_validate_data(self,
                      X,
                      y=None,
                      dtype=None,
                      **kwargs):
    if isinstance(X, pd.DataFrame):
        cols = X.columns
        X_out, y_out = check_X_y(X.values, y, accept_sparse=False, dtype=dtype)
        X_df = pd.DataFrame(X_out, columns=cols)
        _set_feature_metadata(self, X_df)
        return X_df, y_out

    X_out, y_out = check_X_y(X, y, accept_sparse=False, dtype=dtype)
    _set_feature_metadata(self, X_out)
    return X_out, y_out

EnsembleGenerator._validate_data = _eg_validate_data

def _patched_uff_fit(self, X, y=None):
    X_arr = check_array(X, accept_sparse=False)
    self.n_features_in_ = X_arr.shape[1]
    if hasattr(X, "columns"):
        self.feature_names_in_ = np.array(X.columns, dtype=object)

    thr = getattr(self, "threshold", 1)
    if X_arr.shape[0] <= thr:
        self.features_to_keep_ = np.ones(self.n_features_in_, dtype=bool)
    else:
        self.features_to_keep_ = np.array(
            [len(np.unique(X_arr[:, i])) > thr for i in range(self.n_features_in_)]
        )
    self.n_features_out_ = int(self.features_to_keep_.sum())
    return self

def _patched_uff_transform(self, X):
    X_arr = check_array(X, accept_sparse=False)
    return X_arr[:, self.features_to_keep_]

UniqueFeatureFilter.fit = _patched_uff_fit
UniqueFeatureFilter.transform = _patched_uff_transform

def _generic_validate_data(self,
                           X,
                           y=None,
                           reset=True,
                           dtype=None,
                           **kwargs):
    if isinstance(X, pd.DataFrame):
        cols = X.columns
        X_out = check_array(X.values, accept_sparse=False, dtype=dtype)
        X_df = pd.DataFrame(X_out, columns=cols)
        _set_feature_metadata(self, X_df)
        return X_df

    X_out = check_array(X, accept_sparse=False, dtype=dtype)
    _set_feature_metadata(self, X_out)
    return X_out

PreprocessingPipeline._validate_data = _generic_validate_data
CustomStandardScaler._validate_data = _generic_validate_data
OutlierRemover._validate_data = _generic_validate_data

def _ttn_transform_identity(self, X):
    if isinstance(X, pd.DataFrame):
        arr = X.values
    elif isinstance(X, (list, tuple)):
        elems = [e for e in X if e is not None]
        if len(elems) == 0:
            raise ValueError("TransformToNumerical: got empty list/tuple")
        if len(elems) == 1 and hasattr(elems[0], "shape"):
            arr = np.asarray(elems[0])
        else:
            arrs = [np.asarray(e) for e in elems if hasattr(e, "shape")]
            if len(arrs) == 0:
                raise ValueError(f"TransformToNumerical: cannot interpret X={X!r}")
            n_samples = arrs[0].shape[0]
            arrs = [a.reshape(n_samples, -1) for a in arrs]
            arr = np.concatenate(arrs, axis=1)
    else:
        arr = np.asarray(X)
        if arr.ndim == 1 and arr.dtype == object:
            elems = [e for e in arr if e is not None]
            if len(elems) == 0:
                raise ValueError("TransformToNumerical: got empty object array")
            if len(elems) == 1 and hasattr(elems[0], "shape"):
                arr = np.asarray(elems[0])
            else:
                arrs = [np.asarray(e) for e in elems if hasattr(e, "shape")]
                n_samples = arrs[0].shape[0]
                arrs = [a.reshape(n_samples, -1) for a in arrs]
                arr = np.concatenate(arrs, axis=1)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    return arr

TransformToNumerical.transform = _ttn_transform_identity

def make_tabicl_classifier(random_state):
    kwargs = {"random_state": random_state, "verbose": False}
    try:
        sig = inspect.signature(TabICLClassifier)
        if "device" in sig.parameters:
            kwargs["device"] = DEVICE
    except Exception:
        pass
    return TabICLClassifier(**kwargs)

# ============================================================
# Load data
# ============================================================
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)
y_raw = df[TARGET_COL].to_numpy()
X_df  = df.drop(columns=[TARGET_COL])

y_all, class_map, NUM_CLASSES = ensure_multiclass_contiguous(y_raw)

# global standardization to match tuning pipeline
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values).astype(np.float32, copy=False)

X_all = np.asarray(X_scaled, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.int64)

print(f"[DATA] {DATA_FILE} | X={X_all.shape} | C={NUM_CLASSES} | map={class_map}")
print(f"[CUDA] available={torch.cuda.is_available()} | visible={torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"[CUDA] using logical device: {DEVICE} (physical GPU 7 via CUDA_VISIBLE_DEVICES=7)")
    print(f"[CUDA] name: {torch.cuda.get_device_name(0)}")

# ============================================================
# Shared CV splits
# ============================================================
rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
splits = list(rkf.split(X_all, y_all))

# ============================================================
# GO-LR once on FULL X
# ============================================================
t0_total = time.time()

go = GraphFeatureOrdering(
    num_clusters=int(BEST_PARAMS["go_num_clusters"]),
    metric=BEST_PARAMS["go_metric"],
    refine=True,
    direction_select=bool(BEST_PARAMS["go_direction_select"]),
    refine_passes=int(BEST_PARAMS["go_refine_passes"]),
)

t0_go = time.time()
try:
    Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=False)
except RuntimeError:
    cleanup_cuda()
    Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=True)
t1_go = time.time()

print(f"[GO-LR] done in {t1_go - t0_go:.2f}s | len(Pi_star)={len(Pi_star)}")

# ============================================================
# Prepare TabPFN config
# ============================================================
tabpfn_cfg = TabPFN25Config(
    task_type="multiclass",
    num_classes=int(NUM_CLASSES),
    device=DEVICE,
    random_state=int(BEST_PARAMS["tabpfn_seed"]),
)

# ============================================================
# Evaluation loops
# ============================================================
tabpfn_accs, tabpfn_aucs, tabpfn_f1s, tabpfn_times = [], [], [], []
tabicl_accs, tabicl_aucs, tabicl_f1s, tabicl_times = [], [], [], []

for fold_id, (tr_idx, va_idx) in enumerate(splits, start=1):
    X_tr = X_all[tr_idx]
    y_tr = y_all[tr_idx]
    X_va = X_all[va_idx]
    y_va = y_all[va_idx]

    # ---------------- NSC on train only ----------------
    nsc = PIDFSegPCA(
        segmentation=BEST_PARAMS["nsc_segmentation"],
        l_min=int(BEST_PARAMS["nsc_lmin"]),
        m_rule=BEST_PARAMS["nsc_m_rule"],
        gamma=float(BEST_PARAMS["nsc_gamma"]),
        beta=float(BEST_PARAMS["nsc_beta"]),
        tau=float(BEST_PARAMS["nsc_tau"]),
        M_min=int(BEST_PARAMS["nsc_Mmin"]),
        M_max=int(BEST_PARAMS["nsc_Mmax"]),
        assume_standardized=bool(BEST_PARAMS["assume_standardized"]),
        device=DEVICE,
    )

    deltas = None if BEST_PARAMS["nsc_segmentation"] == "uniform" else compute_deltas_adjacent_corr(X_tr, Pi_star)

    X_tr_t = torch.from_numpy(X_tr)
    X_va_t = torch.from_numpy(X_va)

    nsc.configure(
        Pi_star=Pi_star,
        X_train=X_tr_t,
        tau=float(BEST_PARAMS["nsc_tau"]),
        deltas=deltas,
    )

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy().astype(np.float32, copy=False)
    Z_va = nsc.compress(X_va_t, mode="flatten").cpu().numpy().astype(np.float32, copy=False)

    # ---------------- TabPFN-2.5 ----------------
    t0_pf = time.time()
    head = TabPFN25Head(tabpfn_cfg)
    head.fit(Z_tr, y_tr)
    P_pf = head.predict_proba(Z_va)
    pred_pf = np.argmax(P_pf, axis=1)
    acc_pf, auc_pf, f1_pf = compute_multiclass_metrics(y_va, P_pf, pred_pf, NUM_CLASSES)
    t1_pf = time.time()

    tabpfn_accs.append(acc_pf)
    tabpfn_aucs.append(auc_pf)
    tabpfn_f1s.append(f1_pf)
    tabpfn_times.append(t1_pf - t0_pf)

    # ---------------- TabICL ----------------
    t0_ti = time.time()
    Z_tr_df = pd.DataFrame(Z_tr, columns=[f"z_{i}" for i in range(Z_tr.shape[1])])
    Z_va_df = pd.DataFrame(Z_va, columns=[f"z_{i}" for i in range(Z_va.shape[1])])

    clf = make_tabicl_classifier(SEED + fold_id)
    clf.fit(Z_tr_df, y_tr)
    P_ti = clf.predict_proba(Z_va_df)
    pred_ti = clf.predict(Z_va_df)

    if hasattr(P_ti, "values"):
        P_ti = P_ti.values
    else:
        P_ti = np.asarray(P_ti)

    acc_ti, auc_ti, f1_ti = compute_multiclass_metrics(y_va, P_ti, pred_ti, NUM_CLASSES)
    t1_ti = time.time()

    tabicl_accs.append(acc_ti)
    tabicl_aucs.append(auc_ti)
    tabicl_f1s.append(f1_ti)
    tabicl_times.append(t1_ti - t0_ti)

    print(
        f"[Fold {fold_id:02d}/25] "
        f"TabPFN: ACC={acc_pf:.6f} AUC={auc_pf:.6f} F1={f1_pf:.6f} ({t1_pf-t0_pf:.2f}s) || "
        f"TabICL: ACC={acc_ti:.6f} AUC={auc_ti:.6f} F1={f1_ti:.6f} ({t1_ti-t0_ti:.2f}s) || "
        f"Z={Z_tr.shape[1]}"
    )

    cleanup_cuda()

t1_total = time.time()

# ============================================================
# Summary
# ============================================================
def mean_std(x):
    return float(np.mean(x)), float(np.std(x, ddof=1))

pf_acc_m, pf_acc_s = mean_std(tabpfn_accs)
pf_auc_m, pf_auc_s = mean_std(tabpfn_aucs)
pf_f1_m,  pf_f1_s  = mean_std(tabpfn_f1s)
pf_rt = float(np.sum(tabpfn_times))

ti_acc_m, ti_acc_s = mean_std(tabicl_accs)
ti_auc_m, ti_auc_s = mean_std(tabicl_aucs)
ti_f1_m,  ti_f1_s  = mean_std(tabicl_f1s)
ti_rt = float(np.sum(tabicl_times))

print("\n" + "=" * 78)
print("FINAL 5x5 CV RESULTS: Lung multiclass with fixed GO-LR + NSC hyperparameters")
print("=" * 78)
print(f"{'Model':<14} {'Accuracy':<22} {'ROC-AUC':<22} {'Macro-F1':<22} {'Runtime'}")
print("-" * 78)
print(f"{'TabPFN-2.5':<14} {f'{pf_acc_m:.6f} ± {pf_acc_s:.6f}':<22} {f'{pf_auc_m:.6f} ± {pf_auc_s:.6f}':<22} {f'{pf_f1_m:.6f} ± {pf_f1_s:.6f}':<22} {pf_rt:.2f}s")
print(f"{'TabICL':<14} {f'{ti_acc_m:.6f} ± {ti_acc_s:.6f}':<22} {f'{ti_auc_m:.6f} ± {ti_auc_s:.6f}':<22} {f'{ti_f1_m:.6f} ± {ti_f1_s:.6f}':<22} {ti_rt:.2f}s")
print("-" * 78)
print(f"Total wall-clock runtime (including GO-LR + NSC + both models): {t1_total - t0_total:.2f}s ({(t1_total - t0_total)/60.0:.2f} min)")
print("=" * 78)

[DATA] lung_combined_encoded.csv | X=(203, 3312) | C=5 | map={0: 0, 1: 1, 2: 2, 3: 3, 4: 4}
[CUDA] available=True | visible=1
[CUDA] using logical device: cuda (physical GPU 7 via CUDA_VISIBLE_DEVICES=7)
[CUDA] name: NVIDIA TITAN RTX
[GO-LR] done in 524.32s | len(Pi_star)=3312
[Fold 01/25] TabPFN: ACC=1.000000 AUC=1.000000 F1=1.000000 (19.12s) || TabICL: ACC=0.975610 AUC=1.000000 F1=0.967920 (15.18s) || Z=302
[Fold 02/25] TabPFN: ACC=0.975610 AUC=0.997297 F1=0.967920 (18.94s) || TabICL: ACC=0.975610 AUC=0.998099 F1=0.967920 (16.49s) || Z=302
[Fold 03/25] TabPFN: ACC=0.975610 AUC=0.999451 F1=0.956491 (19.87s) || TabICL: ACC=0.951220 AUC=0.994293 F1=0.893103 (18.38s) || Z=302
[Fold 04/25] TabPFN: ACC=1.000000 AUC=1.000000 F1=1.000000 (17.93s) || TabICL: ACC=0.950000 AUC=1.000000 F1=0.764532 (12.25s) || Z=302
[Fold 05/25] TabPFN: ACC=0.925000 AUC=0.984010 F1=0.926013 (17.97s) || TabICL: ACC=0.900000 AUC=0.988746 F1=0.895726 (14.37s) || Z=302
[Fold 06/25] TabPFN: ACC=0.975610 AUC=0.996154 